[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tsilva/aiml-notebooks/blob/main/notebooks/vqvae-mnist.ipynb)

# Vector Quantized Variational Autoencoder (VQ-VAE) on Image Datasets

This notebook trains a VQ-VAE to learn a discrete latent representation of images using a learned codebook, then generates new images by sampling from the discrete latent space. Supports multiple datasets: MNIST, Fashion-MNIST, and CIFAR-10.

Now we'll import the necessary libraries and enable autoreload.

In [ ]:
# Import required libraries
import torch
import torch.nn as nn
import torch.nn.functional as F
import pytorch_lightning as L
import matplotlib.pyplot as plt
import numpy as np
import wandb

# Import shared utilities from local package
from aiml_notebooks import create_dataset, create_dataloaders, create_trainer, plot_image_grid, log_images_to_wandb

# Enable autoreload for hot reloading
%load_ext autoreload
%autoreload 2

print(f"PyTorch: {torch.__version__}")
print(f"Lightning: {L.__version__}")

Next, we'll define all hyperparameters in a CONFIG dictionary and set random seeds for reproducibility.

In [ ]:
# Configuration (Base defaults - can be overridden by papermill parameters)
CONFIG = {
    # Data
    'dataset_id': 'mnist',           # Dataset: 'mnist', 'fashionmnist', or 'cifar10'
    'seed': 42,                      # Random seed for reproducibility
    'train_split': 0.857,            # 60K / 70K = ~0.857 (MNIST standard train size)
    'val_split': 0.143,              # 10K / 70K = ~0.143 (MNIST standard test size)
    'batch_size': 128,               # Number of images per batch
    'num_workers': 4,                # Parallel data loading workers
    
    # Model
    'embedding_dim': 64,             # Dimension of each codebook vector
    'num_embeddings': 512,           # Number of vectors in the codebook
    'hidden_dims': [32, 64],         # Conv layer channels (encoder/decoder)
    'learning_rate': 1e-3,           # Step size for optimizer (0.001)
    
    # Training
    'max_epochs': 50,                # Number of complete passes through data
    'log_every_n_steps': 50,         # How often to log training metrics
    
    # Loss weights
    'commitment_cost': 0.25,         # Weight for commitment loss (encoder committed to embeddings)
    
    # Visualization
    'num_samples': 16,               # Number of images to generate
    
    # Weights & Biases
    'wandb_project': 'vqvae-mnist',  # W&B project name
    'wandb_run_name': None,          # Optional run name (None = auto-generated)
}

# Set random seeds
L.seed_everything(CONFIG['seed'])

Now we'll load the dataset and create data loaders for training and validation.

In [ ]:
# Create dataset using the factory (handles data loading and splitting)
full_dataset, train_dataset, val_dataset = create_dataset(
    dataset_id=CONFIG['dataset_id'],
    splits=[CONFIG['train_split'], CONFIG['val_split']]
)

# Create data loaders using the factory
train_loader, val_loader = create_dataloaders(
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    batch_size=CONFIG['batch_size'],
    num_workers=CONFIG['num_workers'],
    use_collate_fn=False  # Vision datasets don't need padding
)

print(f"Dataset: {CONFIG['dataset_id']}")
print(f"Total samples: {len(full_dataset)}")
print(f"Train samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset)}")

# Get a sample to check image shape
sample_batch = next(iter(train_loader))
sample_image = sample_batch[0][0]
print(f"Batch shape: {sample_batch[0].shape}")
print(f"Image shape: {sample_image.shape}")
print(f"Channels: {sample_image.shape[0]}, Height: {sample_image.shape[1]}, Width: {sample_image.shape[2]}")

Now we'll define the Vector Quantizer layer that learns a discrete codebook of embedding vectors.

In [ ]:
class VectorQuantizer(nn.Module):
    """
    Vector Quantization layer with learnable codebook.
    
    Maps continuous encoder outputs to discrete codebook vectors using
    nearest-neighbor lookup, enabling discrete latent representations.
    
    Args:
        num_embeddings: Size of the codebook (number of discrete codes)
        embedding_dim: Dimension of each codebook vector
        commitment_cost: Weight for commitment loss (default: 0.25)
    """
    def __init__(self, num_embeddings, embedding_dim, commitment_cost=0.25):
        super().__init__()
        self.embedding_dim = embedding_dim
        self.num_embeddings = num_embeddings
        self.commitment_cost = commitment_cost
        
        # Learnable codebook: (num_embeddings, embedding_dim)
        self.embedding = nn.Embedding(num_embeddings, embedding_dim)
        self.embedding.weight.data.uniform_(-1/num_embeddings, 1/num_embeddings)
    
    def forward(self, z):
        """
        Quantize continuous latent vectors to discrete codebook entries.
        
        Args:
            z: Encoder output (batch_size, embedding_dim, height, width)
        
        Returns:
            quantized: Quantized latent (same shape as z)
            loss: VQ loss (codebook + commitment)
            perplexity: Measure of codebook usage (higher = more codes used)
            encodings: One-hot encoded indices (batch_size*H*W, num_embeddings)
        """
        # Flatten spatial dimensions: (B, C, H, W) -> (B*H*W, C)
        z = z.permute(0, 2, 3, 1).contiguous()
        z_flattened = z.view(-1, self.embedding_dim)
        
        # Calculate distances to all codebook entries
        # ||z - e||^2 = ||z||^2 + ||e||^2 - 2*z*e
        distances = (
            torch.sum(z_flattened**2, dim=1, keepdim=True) +
            torch.sum(self.embedding.weight**2, dim=1) -
            2 * torch.matmul(z_flattened, self.embedding.weight.t())
        )
        
        # Find nearest codebook entry for each encoder output
        encoding_indices = torch.argmin(distances, dim=1)
        encodings = F.one_hot(encoding_indices, self.num_embeddings).float()
        
        # Quantize by looking up the nearest codebook entry
        quantized = torch.matmul(encodings, self.embedding.weight)
        quantized = quantized.view(z.shape)
        
        # Calculate VQ losses:
        # 1. Codebook loss: move codebook entries towards encoder outputs
        e_latent_loss = F.mse_loss(quantized.detach(), z)
        # 2. Commitment loss: encourage encoder to commit to codebook entries
        q_latent_loss = F.mse_loss(quantized, z.detach())
        loss = q_latent_loss + self.commitment_cost * e_latent_loss
        
        # Straight-through estimator: copy gradients from decoder to encoder
        quantized = z + (quantized - z).detach()
        
        # Calculate perplexity (measure of codebook usage)
        avg_probs = torch.mean(encodings, dim=0)
        perplexity = torch.exp(-torch.sum(avg_probs * torch.log(avg_probs + 1e-10)))
        
        # Return to original shape: (B*H*W, C) -> (B, C, H, W)
        quantized = quantized.permute(0, 3, 1, 2).contiguous()
        
        return quantized, loss, perplexity, encodings

Now we'll define the VQ-VAE model that automatically adapts to different image sizes and channel counts.

In [ ]:
# Vector Quantized Variational Autoencoder with adaptive architecture
class VQVAE(L.LightningModule):
    def __init__(
        self,
        in_channels=1,
        image_size=28,
        embedding_dim=64, 
        num_embeddings=512, 
        hidden_dims=[32, 64], 
        learning_rate=1e-3,
        commitment_cost=0.25
    ):
        super().__init__()
        self.save_hyperparameters()
        
        # Calculate spatial size after convolutions
        self.spatial_size = image_size // 4
        
        # Encoder: adapts to input channels and image size
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, hidden_dims[0], kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(hidden_dims[0], hidden_dims[1], kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(hidden_dims[1], embedding_dim, kernel_size=1),  # Project to embedding_dim
        )
        
        # Vector Quantizer with learnable codebook
        self.vq = VectorQuantizer(
            num_embeddings=num_embeddings,
            embedding_dim=embedding_dim,
            commitment_cost=commitment_cost
        )
        
        # Decoder: adapts to output channels
        self.decoder = nn.Sequential(
            nn.Conv2d(embedding_dim, hidden_dims[1], kernel_size=1),  # Project from embedding_dim
            nn.ConvTranspose2d(hidden_dims[1], hidden_dims[0], kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(hidden_dims[0], in_channels, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.Sigmoid(),  # Output in [0, 1] range
        )
    
    def encode(self, x):
        """Encode image to continuous latent representation."""
        return self.encoder(x)
    
    def decode(self, z):
        """Decode quantized latent to image."""
        return self.decoder(z)
    
    def forward(self, x):
        z = self.encode(x)
        quantized, vq_loss, perplexity, _ = self.vq(z)
        recon = self.decode(quantized)
        return recon, vq_loss, perplexity
    
    def training_step(self, batch, batch_idx):
        x, _ = batch
        recon, vq_loss, perplexity = self(x)
        
        # Reconstruction loss (binary cross-entropy)
        recon_loss = F.binary_cross_entropy(recon, x)
        
        # Total loss = reconstruction + VQ losses
        loss = recon_loss + vq_loss
        
        self.log('train_loss', loss, prog_bar=True)
        self.log('train_recon_loss', recon_loss)
        self.log('train_vq_loss', vq_loss)
        self.log('train_perplexity', perplexity)
        
        return loss
    
    def validation_step(self, batch, batch_idx):
        x, _ = batch
        recon, vq_loss, perplexity = self(x)
        
        recon_loss = F.binary_cross_entropy(recon, x)
        loss = recon_loss + vq_loss
        
        self.log('val_loss', loss, prog_bar=True)
        self.log('val_recon_loss', recon_loss)
        self.log('val_vq_loss', vq_loss)
        self.log('val_perplexity', perplexity)
        
        return loss
    
    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.learning_rate)
    
    @torch.no_grad()
    def sample(self, num_samples=16):
        """Generate new images by sampling random codebook vectors."""
        self.eval()
        # Sample random codebook indices
        indices = torch.randint(0, self.hparams.num_embeddings, (num_samples, self.spatial_size, self.spatial_size), device=self.device)
        # Look up codebook vectors
        quantized = self.vq.embedding(indices)  # (num_samples, spatial_size, spatial_size, embedding_dim)
        quantized = quantized.permute(0, 3, 1, 2).contiguous()  # (num_samples, embedding_dim, spatial_size, spatial_size)
        # Decode to images
        samples = self.decode(quantized)
        return samples

# Initialize model with dataset-specific parameters
model = VQVAE(
    in_channels=sample_image.shape[0],  # 1 for grayscale, 3 for RGB
    image_size=sample_image.shape[1],   # 28 for MNIST/Fashion-MNIST, 32 for CIFAR-10
    embedding_dim=CONFIG['embedding_dim'],
    num_embeddings=CONFIG['num_embeddings'],
    hidden_dims=CONFIG['hidden_dims'],
    learning_rate=CONFIG['learning_rate'],
    commitment_cost=CONFIG['commitment_cost'],
)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Input: {sample_image.shape[0]} channels, {sample_image.shape[1]}x{sample_image.shape[2]} pixels")
print(f"Codebook: {CONFIG['num_embeddings']} vectors of dimension {CONFIG['embedding_dim']}")
print(f"Latent spatial size: {model.spatial_size}x{model.spatial_size}")

Now we'll train the model using PyTorch Lightning's Trainer with W&B logging.

In [ ]:
# Train the model (W&B logger created automatically)
trainer = create_trainer(
    max_epochs=CONFIG['max_epochs'],
    log_every_n_steps=CONFIG['log_every_n_steps'],
    wandb_project=CONFIG['wandb_project'],
    wandb_run_name=CONFIG['wandb_run_name'],
    wandb_config=CONFIG,
    model=model
)
trainer.fit(model, train_loader, val_loader)

Finally, we'll visualize reconstructions and analyze the learned discrete codebook usage.

In [ ]:
# Get test images for each class (0-9)
model.eval()

# Collect multiple examples of each class from validation set
class_examples = {i: [] for i in range(10)}
for images, labels in val_loader:
    for img, label in zip(images, labels):
        class_id = label.item()
        if len(class_examples[class_id]) < 100:  # Collect 100 examples per class
            class_examples[class_id].append(img)
    if all(len(examples) >= 100 for examples in class_examples.values()):
        break

# Get one representative image per class for display
test_images = torch.stack([class_examples[i][0] for i in range(10)])

with torch.no_grad():
    # Reconstruct the representative images
    test_images_gpu = test_images.to(model.device)
    reconstructions, _, _ = model(test_images_gpu)
    
    # Generate random samples from the codebook
    samples = model.sample(num_samples=30)

# Move to CPU for plotting
test_images = test_images.cpu()
reconstructions = reconstructions.cpu()
samples = samples.cpu()

# Stack images: [Original (10) | Reconstruction (10) | Samples (30 in 3 rows)]
all_images = torch.cat([test_images, reconstructions, samples], dim=0)

# Create labels
col_labels = [f'Class {i}' for i in range(10)]
row_labels = ['Original', 'Reconstructed', 'Random Sample #1', 'Random Sample #2', 'Random Sample #3']

# Determine colormap based on dataset
cmap = None if sample_image.shape[0] == 3 else 'gray'  # RGB uses None, grayscale uses 'gray'

# Plot using shared utility (5 rows × 10 columns)
fig = plot_image_grid(
    images=all_images,
    nrows=5,
    ncols=10,
    row_labels=row_labels,
    col_labels=col_labels,
    figsize=(12, 6),
    cmap=cmap
)
plt.show()

# Log to W&B
log_images_to_wandb(fig, 'reconstructions')

# Dataset-specific class names
class_names = {
    'mnist': ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9'],
    'fashionmnist': ['T-shirt', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot'],
    'cifar10': ['Airplane', 'Car', 'Bird', 'Cat', 'Deer', 'Dog', 'Frog', 'Horse', 'Ship', 'Truck']
}
dataset_classes = class_names.get(CONFIG['dataset_id'], [f'Class {i}' for i in range(10)])

print("\n" + "="*50)
print("VQ-VAE RESULTS")
print("="*50)
print(f"Dataset: {CONFIG['dataset_id'].upper()}")
print(f"Classes: {', '.join(dataset_classes)}")
print(f"Row 1: Original test images")
print(f"Row 2: Reconstructions (VQ-VAE output)")
print(f"Rows 3-5: Random samples from discrete codebook")
print(f"\nCodebook: {CONFIG['num_embeddings']} discrete codes")

Let's analyze the codebook usage to see which discrete codes are being used.

In [ ]:
# Analyze codebook usage across validation set
model.eval()
all_encodings = []

with torch.no_grad():
    for images, _ in val_loader:
        images = images.to(model.device)
        z = model.encode(images)
        _, _, _, encodings = model.vq(z)
        all_encodings.append(encodings)

# Concatenate all encodings
all_encodings = torch.cat(all_encodings, dim=0)

# Calculate usage statistics
codebook_usage = all_encodings.sum(dim=0).cpu().numpy()
total_codes = len(codebook_usage)
used_codes = (codebook_usage > 0).sum()
usage_percentage = (used_codes / total_codes) * 100

print("\n" + "="*50)
print("CODEBOOK ANALYSIS")
print("="*50)
print(f"Total codebook size: {total_codes}")
print(f"Codes actually used: {used_codes} ({usage_percentage:.1f}%)")
print(f"Unused codes: {total_codes - used_codes}")

# Plot codebook usage histogram
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(range(total_codes), codebook_usage)
ax.set_xlabel('Codebook Index')
ax.set_ylabel('Usage Count')
ax.set_title(f'VQ-VAE Codebook Usage ({used_codes}/{total_codes} codes active)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Log to W&B
log_images_to_wandb(fig, 'codebook_usage')

# Log statistics to W&B
wandb.log({
    'codebook_usage_percentage': usage_percentage,
    'codebook_used_codes': used_codes,
    'codebook_unused_codes': total_codes - used_codes,
})